# Suivi d'expériences NumPyro avec MLflow

Ce notebook montre comment **tracer dans MLflow des ajustements (*fits*) de modèles bayésiens NumPyro**, sur le jeu de données d'attrition des collaborateurs (`data/employees.csv`).

Contrairement à scikit-learn, il n'existe pas d'`autolog` pour NumPyro : on enregistre donc explicitement, pour chaque run,

| Quoi | Où dans MLflow |
|---|---|
| modèle, priors, réglages de l'échantillonneur NUTS, graine | **paramètres** |
| diagnostics de convergence (R-hat, ESS, divergences), durée | **métriques** |
| qualité prédictive (ELPD-LOO, AUC, log-loss, F1 sur le jeu de test) | **métriques** |
| code source du modèle, résumé du postérieur, échantillons (NetCDF), *trace plot* | **artefacts** |
| versions de `numpyro` / `jax` | **tags** |

**Prérequis**

```bash
uv add numpyro arviz netcdf4                               # dépendances supplémentaires (netcdf4 : écriture NetCDF)
uv run mlflow server --host 127.0.0.1 --port 5000   # dans un terminal dédié
```

## 1. Imports et configuration

In [2]:
import inspect
import os
import tempfile
import time
from pathlib import Path

import arviz as az
import jax
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import numpyro
import numpyro.distributions as dist
import pandas as pd
import xarray as xr
from jax import random
from numpyro.diagnostics import summary
from numpyro.infer import MCMC, NUTS, Predictive
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from training.data import load_data

In [3]:
# Adresse du serveur MLflow : variable d'environnement si elle existe, sinon serveur local
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT = "employee_attrition_numpyro"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)

2026/10/06 06:50:24 INFO mlflow.tracking.fluent: Experiment with name 'employee_attrition_numpyro' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1791262224314, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1791262224314, lifecycle_stage='active', name='employee_attrition_numpyro', tags={}, trace_location=None, workspace='default'>

## 2. Données

On standardise les variables numériques (indispensable pour que les priors aient un sens et que NUTS échantillonne bien) et on encode les variables catégorielles en indicatrices.

Deux jeux de variables sont préparés, un par famille de modèles :

- `pooled` : la ville est une variable explicative comme les autres (indicatrices) ;
- `hierarchical` : la ville n'est pas dans `X`, elle sert d'index de groupe pour une ordonnée à l'origine par ville.

In [4]:
df = load_data("../data/employees.csv")

TARGET = "LeaveOrNot"
NUMERICAL = ["JoiningYear", "Age", "ExperienceInCurrentDomain"]
CATEGORICAL = ["Education", "PaymentTier", "Gender", "EverBenched"]

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df[TARGET], random_state=42
)
y_train = train_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()


def make_preprocessor(categorical: list[str]) -> ColumnTransformer:
    return ColumnTransformer(
        [
            ("num", StandardScaler(), NUMERICAL),
            ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical),
        ],
        verbose_feature_names_out=False,
    )


PREPROCESSORS = {
    "pooled": make_preprocessor(CATEGORICAL + ["City"]).fit(train_df),
    "hierarchical": make_preprocessor(CATEGORICAL).fit(train_df),
}
CITIES = sorted(train_df["City"].unique())


def model_inputs(variant: str, data: pd.DataFrame) -> dict:
    # Arguments attendus par le modèle NumPyro (hors y et priors).
    inputs = {"X": PREPROCESSORS[variant].transform(data)}
    if variant == "hierarchical":
        inputs["city"] = data["City"].map({c: i for i, c in enumerate(CITIES)}).to_numpy()
        inputs["n_cities"] = len(CITIES)
    return inputs


{v: list(p.get_feature_names_out()) for v, p in PREPROCESSORS.items()}

{'pooled': ['JoiningYear',
  'Age',
  'ExperienceInCurrentDomain',
  'Education_Masters',
  'Education_PHD',
  'PaymentTier_2',
  'PaymentTier_3',
  'Gender_Male',
  'EverBenched_Yes',
  'City_New Delhi',
  'City_Pune'],
 'hierarchical': ['JoiningYear',
  'Age',
  'ExperienceInCurrentDomain',
  'Education_Masters',
  'Education_PHD',
  'PaymentTier_2',
  'PaymentTier_3',
  'Gender_Male',
  'EverBenched_Yes']}

## 3. Modèles NumPyro

Deux régressions logistiques bayésiennes. `prior_scale` (écart-type du prior sur les coefficients) est l'hyperparamètre que l'on fera varier d'un run à l'autre.

Quand `y=None` (prédiction), le modèle expose la probabilité de départ `p` comme site déterministe ; pendant l'ajustement elle n'est pas stockée, pour ne pas alourdir le postérieur.

In [5]:
def logistic_pooled(X, y=None, prior_scale=1.0):
    # Régression logistique : P(départ) = sigmoid(alpha + X @ beta).
    n, k = X.shape
    alpha = numpyro.sample("alpha", dist.Normal(0.0, 2.0))
    beta = numpyro.sample("beta", dist.Normal(0.0, prior_scale).expand([k]).to_event(1))
    logits = alpha + X @ beta
    if y is None:
        numpyro.deterministic("p", jax.nn.sigmoid(logits))
    with numpyro.plate("obs", n):
        numpyro.sample("y", dist.Bernoulli(logits=logits), obs=y)


def logistic_hierarchical(X, city, n_cities, y=None, prior_scale=1.0):
    # Régression logistique avec ordonnée à l'origine par ville (pooling partiel).
    n, k = X.shape
    mu = numpyro.sample("mu", dist.Normal(0.0, 2.0))
    sigma_city = numpyro.sample("sigma_city", dist.HalfNormal(1.0))
    with numpyro.plate("cities", n_cities):
        z_city = numpyro.sample("z_city", dist.Normal(0.0, 1.0))  # paramétrisation non centrée
    alpha_city = numpyro.deterministic("alpha_city", mu + sigma_city * z_city)
    beta = numpyro.sample("beta", dist.Normal(0.0, prior_scale).expand([k]).to_event(1))
    logits = alpha_city[city] + X @ beta
    if y is None:
        numpyro.deterministic("p", jax.nn.sigmoid(logits))
    with numpyro.plate("obs", n):
        numpyro.sample("y", dist.Bernoulli(logits=logits), obs=y)


MODELS = {"pooled": logistic_pooled, "hierarchical": logistic_hierarchical}

## 4. Fonctions d'enregistrement

`run_experiment` encapsule **un fit = un run MLflow**. Les fonctions auxiliaires calculent ce qu'on y enregistre.

In [6]:
def posterior_summary(mcmc: MCMC) -> pd.DataFrame:
    # Résumé du postérieur (moyenne, écart-type, quantiles, n_eff, r_hat), une ligne par paramètre scalaire.
    stats = summary(mcmc.get_samples(group_by_chain=True))
    rows = []
    for site, site_stats in stats.items():
        shape = np.shape(site_stats["mean"])
        for idx in np.ndindex(shape):
            name = site if not shape else f"{site}[{','.join(map(str, idx))}]"
            rows.append({"param": name} | {k: float(np.asarray(v)[idx]) for k, v in site_stats.items()})
    return pd.DataFrame(rows).set_index("param")


def loo_metrics(idata) -> dict[str, float]:
    # ELPD par validation croisée leave-one-out (PSIS-LOO) : plus il est élevé, mieux le modèle prédit.
    loo = az.loo(idata)

    def field(*names):  # les noms d'attributs diffèrent entre ArviZ 0.x et 1.x
        return float(next(getattr(loo, n) for n in names if hasattr(loo, n)))

    return {
        "elpd_loo": field("elpd", "elpd_loo"),
        "elpd_loo_se": field("se"),
        "p_loo": field("p", "p_loo"),
    }


def trace_figure(mcmc: MCMC, site: str = "beta", labels: list[str] | None = None):
    # Trace plot minimal : distribution marginale et trace par chaîne pour chaque composante du site.
    chains = np.asarray(mcmc.get_samples(group_by_chain=True)[site])  # (chaînes, tirages, k)
    chains = chains.reshape(*chains.shape[:2], -1)
    k = chains.shape[-1]
    fig, axes = plt.subplots(k, 2, figsize=(10, 1.6 * k), squeeze=False)
    for j in range(k):
        for c in range(chains.shape[0]):
            axes[j, 0].hist(chains[c, :, j], bins=30, histtype="step", density=True)
            axes[j, 1].plot(chains[c, :, j], lw=0.5)
        axes[j, 0].set_ylabel(labels[j] if labels else f"{site}[{j}]", rotation=0, ha="right")
        axes[j, 0].set_yticks([])
    axes[0, 0].set_title("Distribution marginale")
    axes[0, 1].set_title("Trace par chaîne")
    fig.tight_layout()
    return fig

In [7]:
def run_experiment(
    variant: str,
    prior_scale: float = 1.0,
    num_warmup: int = 500,
    num_samples: int = 500,
    num_chains: int = 4,
    target_accept_prob: float = 0.8,
    seed: int = 0,
) -> dict:
    # Ajuste un modèle NumPyro par NUTS et enregistre le fit dans un run MLflow.
    model = MODELS[variant]
    train_inputs = model_inputs(variant, train_df)
    test_inputs = model_inputs(variant, test_df)

    with mlflow.start_run(run_name=f"{variant}-scale{prior_scale}") as run:
        # --- Paramètres, tags et code du modèle
        mlflow.set_tags({"numpyro": numpyro.__version__, "jax": jax.__version__, "inference": "NUTS"})
        mlflow.log_params(
            {
                "model": variant,
                "prior_scale": prior_scale,
                "num_warmup": num_warmup,
                "num_samples": num_samples,
                "num_chains": num_chains,
                "target_accept_prob": target_accept_prob,
                "seed": seed,
                "n_train": len(train_df),
                "n_features": train_inputs["X"].shape[1],
            }
        )
        mlflow.log_text(inspect.getsource(model), "model_source.py")

        # --- Ajustement
        mcmc = MCMC(
            NUTS(model, target_accept_prob=target_accept_prob),
            num_warmup=num_warmup,
            num_samples=num_samples,
            num_chains=num_chains,
            chain_method="sequential",
            progress_bar=False,
        )
        start = time.perf_counter()
        mcmc.run(
            random.PRNGKey(seed),
            **train_inputs,
            y=y_train,
            prior_scale=prior_scale,
            extra_fields=("diverging",),
        )
        mlflow.log_metric("fit_duration_s", time.perf_counter() - start)

        # --- Diagnostics de convergence
        summary_df = posterior_summary(mcmc)
        mlflow.log_metrics(
            {
                "divergences": int(mcmc.get_extra_fields()["diverging"].sum()),
                "r_hat_max": float(summary_df["r_hat"].max()),
                "ess_min": float(summary_df["n_eff"].min()),
            }
        )

        # --- Qualité prédictive : LOO sur l'apprentissage, métriques classiques sur le test
        idata = az.from_numpyro(mcmc, log_likelihood=True)
        mlflow.log_metrics(loo_metrics(idata))

        predictive = Predictive(model, mcmc.get_samples(), return_sites=["p"])
        p_draws = predictive(random.PRNGKey(seed + 1), **test_inputs, prior_scale=prior_scale)["p"]
        p_test = np.asarray(p_draws.mean(axis=0))  # moyenne postérieure de P(départ)
        mlflow.log_metrics(
            {
                "test_auc": roc_auc_score(y_test, p_test),
                "test_log_loss": log_loss(y_test, p_test),
                "test_f1": f1_score(y_test, p_test > 0.5),
            }
        )

        # --- Artefacts : résumé, échantillons du postérieur, trace plot
        with tempfile.TemporaryDirectory() as tmp:
            summary_df.to_csv(Path(tmp) / "summary.csv")
            idata.posterior.to_netcdf(Path(tmp) / "posterior.nc")  # sans la log-vraisemblance (volumineuse)
            mlflow.log_artifacts(tmp, artifact_path="posterior")

        fig = trace_figure(mcmc, "beta", list(PREPROCESSORS[variant].get_feature_names_out()))
        mlflow.log_figure(fig, "plots/trace_beta.png")
        plt.close(fig)

        return {"run_id": run.info.run_id, "variant": variant, "mcmc": mcmc, "summary": summary_df}

## 5. Lancer une série d'expériences

Six runs : deux modèles × trois échelles de prior. Chaque run compile puis échantillonne 4 chaînes ; compter quelques dizaines de secondes par run sur CPU.

In [9]:
results = {}
for variant in MODELS:
    for prior_scale in [0.1, 1.0, 10.0]:
        res = run_experiment(variant, prior_scale=prior_scale)
        results[res["run_id"]] = res
        print(f"{variant:<13} prior_scale={prior_scale:<5} -> run {res['run_id']}")

🏃 View run pooled-scale0.1 at: http://127.0.0.1:5000/#/experiments/4/runs/fa0397f5a8594ac09df2027d10a8e812
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
pooled        prior_scale=0.1   -> run fa0397f5a8594ac09df2027d10a8e812
🏃 View run pooled-scale1.0 at: http://127.0.0.1:5000/#/experiments/4/runs/875db6e0f8ee49fe969ee78a20b43406
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
pooled        prior_scale=1.0   -> run 875db6e0f8ee49fe969ee78a20b43406
🏃 View run pooled-scale10.0 at: http://127.0.0.1:5000/#/experiments/4/runs/fae10c6fba414012924bf41a70f7f127
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
pooled        prior_scale=10.0  -> run fae10c6fba414012924bf41a70f7f127
🏃 View run hierarchical-scale0.1 at: http://127.0.0.1:5000/#/experiments/4/runs/6aef90ce7d6740e0993bd792f3e54181
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
hierarchical  prior_scale=0.1   -> run 6aef90ce7d6740e0993bd792f3e54181
🏃 View run hierarchical-scale1.0 at: 

## 6. Comparer les runs

La même comparaison est disponible dans l'interface MLflow (expérience `employee_attrition_numpyro`). Avant de comparer les performances, **vérifier la convergence** : `r_hat_max` proche de 1 (< 1,01), `ess_min` de quelques centaines au moins, `divergences` à 0.

In [10]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["metrics.elpd_loo DESC"])
columns = [
    "tags.mlflow.runName",
    "metrics.elpd_loo",
    "metrics.elpd_loo_se",
    "metrics.test_auc",
    "metrics.test_log_loss",
    "metrics.r_hat_max",
    "metrics.ess_min",
    "metrics.divergences",
    "metrics.fit_duration_s",
    "run_id",
]
runs[columns].round(3)

,tags.mlflow.runName,metrics.elpd_loo,metrics.elpd_loo_se,metrics.test_auc,metrics.test_log_loss,metrics.r_hat_max,metrics.ess_min,metrics.divergences,metrics.fit_duration_s,run_id
0,hierarchical-scale1.0,-2108.081,28.392,0.747,0.554,1.003,552.681,2.0,11.606,5f77b215701c47caa18f48ec9854627b
1,hierarchical-scale10.0,-2108.351,28.465,0.747,0.554,1.003,458.632,1.0,11.945,8411b70489fa4b82aaa23de1603487a6
2,pooled-scale1.0,-2108.438,28.422,0.747,0.554,1.003,959.611,0.0,4.848,875db6e0f8ee49fe969ee78a20b43406
3,pooled-scale10.0,-2108.481,28.520,0.747,0.554,1.001,1091.418,0.0,4.963,fae10c6fba414012924bf41a70f7f127
4,hierarchical-scale0.1,-2126.242,24.822,0.739,0.561,1.011,358.388,2.0,13.106,6aef90ce7d6740e0993bd792f3e54181
5,pooled-scale0.1,-2132.100,23.639,0.740,0.565,1.002,1452.537,0.0,3.863,fa0397f5a8594ac09df2027d10a8e812
6,pooled-scale0.1,-2132.100,23.639,0.740,0.565,1.002,1452.537,0.0,7.529,bdef3bfd2f7f49cf8b425f311d017c7d


## 7. Recharger le postérieur d'un run

Les échantillons enregistrés en NetCDF permettent de reprendre l'analyse plus tard (ou sur une autre machine) sans refaire le fit.

In [11]:
best_run_id = runs.loc[0, "run_id"]

summary_path = mlflow.artifacts.download_artifacts(run_id=best_run_id, artifact_path="posterior/summary.csv")
display(pd.read_csv(summary_path, index_col="param").round(3))

posterior_path = mlflow.artifacts.download_artifacts(run_id=best_run_id, artifact_path="posterior/posterior.nc")
posterior = xr.open_dataset(posterior_path)
posterior

,mean,std,median,5.0%,95.0%,n_eff,r_hat
param,,,,,,,
alpha_city[0],-0.287,0.159,-0.286,-0.535,-0.016,1523.404,1.000
alpha_city[1],-0.925,0.177,-0.926,-1.234,-0.645,1598.578,1.000
alpha_city[2],0.222,0.160,0.225,-0.037,0.480,1552.041,0.999
beta[0],0.393,0.038,0.393,0.330,0.455,2982.286,1.000
beta[1],-0.140,0.039,-0.140,-0.204,-0.076,2907.826,1.001
beta[2],-0.054,0.037,-0.054,-0.113,0.007,2849.850,0.999
beta[3],0.678,0.109,0.680,0.508,0.855,2453.403,0.999
beta[4],0.143,0.204,0.144,-0.190,0.462,2116.853,0.999
beta[5],0.508,0.168,0.505,0.250,0.791,1484.414,1.000


<xarray.Dataset> Size: 140kB
Dimensions:           (chain: 4, draw: 500, alpha_city_dim_0: 3, beta_dim_0: 9,
                       cities: 3)
Coordinates:
  * chain             (chain) int64 32B 0 1 2 3
  * draw              (draw) int64 4kB 0 1 2 3 4 5 6 ... 494 495 496 497 498 499
  * alpha_city_dim_0  (alpha_city_dim_0) int64 24B 0 1 2
  * beta_dim_0        (beta_dim_0) int64 72B 0 1 2 3 4 5 6 7 8
  * cities            (cities) int64 24B 0 1 2
Data variables:
    alpha_city        (chain, draw, alpha_city_dim_0) float32 24kB ...
    beta              (chain, draw, beta_dim_0) float32 72kB ...
    mu                (chain, draw) float32 8kB ...
    sigma_city        (chain, draw) float32 8kB ...
    z_city            (chain, draw, cities) float32 24kB ...
Attributes:
    created_at:                 2026-10-06T04:57:26.473988+00:00
    creation_library:           ArviZ
    creation_library_version:   1.3.1
    creation_library_language:  Python
    inference_library:          numpyro
    inference_library_version:  0.22.0
    sample_dims:                ['chain', 'draw']

## 8. (Option) Enregistrer un modèle servable

Pour passer par le *Model Registry* ou `mlflow models serve`, on emballe le meilleur modèle `pooled` dans un modèle `pyfunc` : le préprocesseur et les tirages du postérieur sont stockés comme artefacts, et `predict` renvoie la probabilité de départ **avec son intervalle de crédibilité à 90 %**, ce qu'un modèle classique ne fournit pas.

In [12]:
import joblib


class NumpyroLogisticModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.preprocessor = joblib.load(context.artifacts["preprocessor"])
        posterior = np.load(context.artifacts["posterior"])
        self.alpha, self.beta = posterior["alpha"], posterior["beta"]

    def predict(self, context, model_input, params=None):
        X = self.preprocessor.transform(model_input)
        p = 1 / (1 + np.exp(-(self.alpha[:, None] + self.beta @ X.T)))  # (tirages, lignes)
        return pd.DataFrame(
            {
                "proba_depart": p.mean(axis=0),
                "p05": np.quantile(p, 0.05, axis=0),
                "p95": np.quantile(p, 0.95, axis=0),
            }
        )


# Meilleur run "pooled" parmi ceux lancés dans cette session
session_runs = runs[runs["run_id"].isin(results)]
best_pooled_id = session_runs.loc[session_runs["params.model"] == "pooled", "run_id"].iloc[0]
samples = results[best_pooled_id]["mcmc"].get_samples()
features = df.drop(columns=[TARGET])

with tempfile.TemporaryDirectory() as tmp, mlflow.start_run(run_id=best_pooled_id):
    joblib.dump(PREPROCESSORS["pooled"], Path(tmp) / "preprocessor.joblib")
    np.savez(Path(tmp) / "posterior.npz", alpha=np.asarray(samples["alpha"]), beta=np.asarray(samples["beta"]))
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=NumpyroLogisticModel(),
        artifacts={
            "preprocessor": str(Path(tmp) / "preprocessor.joblib"),
            "posterior": str(Path(tmp) / "posterior.npz"),
        },
        input_example=features.head(3),
    )

loaded = mlflow.pyfunc.load_model(model_info.model_uri)
loaded.predict(test_df.drop(columns=[TARGET]).head()).round(3)

c:\Users\MIGNOT\Documents\Formation\MLOps\mlops-training\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/10/06 07:02:39 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/10/06 07:02:39 INFO mlflow.pyfunc: Inferring model signature from input example
c:\Users\MIGNOT\Documents\Formation\MLOps\mlops-training\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contain

2026/10/06 07:02:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run pooled-scale1.0 at: http://127.0.0.1:5000/#/experiments/4/runs/875db6e0f8ee49fe969ee78a20b43406
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


,proba_depart,p05,p95
0,0.658,0.612,0.703
1,0.275,0.244,0.308
2,0.107,0.092,0.122
3,0.117,0.096,0.140
4,0.271,0.240,0.305


Le modèle peut ensuite être enregistré dans le registre, comme au TP 1.6 :

```python
mlflow.register_model(model_info.model_uri, name="employee-attrition-numpyro")
```